# Stage C — All Experiments (LLM precision filter)

Batched LLM judge over the funnel's survivors, on **Yandex paid models**. Decides the model/batch, then measures the real precision lift on **SAL**.

## What Stage C is

The funnel's **Stage A** (section-aware localize) + **Stage B** (cue rules + frozen head) is
**recall-first and cheap**, but it leaks false positives: prior-work critiques ("existing methods
cannot…"), contribution claims ("we propose…"), acknowledgments, formulas. **Stage C** asks an LLM,
per surviving candidate, *"is this really the authors' own gap?"* and drops the No's.

Because A+B already cut ~98% of sentences, the LLM sees only ~6 sentences/paper — so LLM-grade
precision costs ~cents per thousands of papers, not the ~$4000/1M of sending whole papers to an LLM.

- Implementation: **`LLMGapFilter`** (imported from `gap2idea`), batched judge, strict-JSON output.
- Prompt mode **`validate_rct`** (biomedical, keep-biased) for the SAL/RCT domain below.
- Backend: **Yandex paid models** (they honor strict `json_schema` — the free tier does not).

> **Key principle:** Stage C is a *precision* filter. It only **improves F1 where there are true
> negatives to remove** — so we evaluate on **SAL** (has negatives), not on positives-only gold
> (where a filter can only cost recall).

In [ ]:
# ===== Stage C setup: Yandex paid models via the repo's LLM layer =====
import os, sys, time, csv, io, json, urllib.request, random
from pathlib import Path
import numpy as np, pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "gap2idea").exists())
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "scripts" / "bench"))
from dotenv import load_dotenv; load_dotenv(ROOT / ".env")
os.environ["LLM_PROVIDER"] = "yandex"                       # use Yandex paid models

from gap2idea.pipeline.gap_llm_filter import LLMGapFilter, SYSTEM_VALIDATE_RCT, BATCH_CHUNK
from gap2idea.pipeline.llm import get_llm_client, active_provider

FOLDER = os.environ["YANDEX_FOLDER_ID"].strip()
def yuri(name): return f"gpt://{FOLDER}/{name}/latest"     # full URI passes through the client untouched
CLIENT = get_llm_client("yandex")
print("provider:", active_provider(), "| default batch chunk:", BATCH_CHUNK)

In [ ]:
# The exact judge prompt (validate_rct) — shown for transparency
print(SYSTEM_VALIDATE_RCT[:900], "...")

## Experiment 1 — which Yandex model + batch size? (small labeled SAL sample)

A precision judge is an easy YES/NO task, so we probe **all candidate models × batch sizes** on a
small labeled sample (true limitations + true negatives from SAL test) and rank by F1, calls, speed.
This is cheap and decides the default used everywhere below.

In [ ]:
# small labeled sample from SAL test (limitation vs not), balanced
def load_sal_raw(split):
    RAW = f"https://raw.githubusercontent.com/MengfeiLan/SAL_Type_Classification/main/data/{split}.csv"
    raw = urllib.request.urlopen(urllib.request.Request(RAW, headers={"User-Agent":"x"}), timeout=60).read().decode("utf-8","replace")
    s, y = [], []
    for r in csv.DictReader(io.StringIO(raw)):
        sent = (r.get("sentences") or "").strip()
        if len(sent.split()) < 4: continue
        s.append(sent); y.append(0 if (r.get("category_spans","[0]") or "[0]").strip()=="[0]" else 1)
    return s, np.array(y)

_s, _y = load_sal_raw("test")
pos = [a for a,b in zip(_s,_y) if b==1]; neg = [a for a,b in zip(_s,_y) if b==0]
random.seed(0); random.shuffle(pos); random.shuffle(neg)
N_POS = N_NEG = 30
sample = [(x,1) for x in pos[:N_POS]] + [(x,0) for x in neg[:N_NEG]]
random.shuffle(sample)
S = [a for a,_ in sample]; G = [b for _,b in sample]
print(f"probe sample: {sum(G)} limitations + {len(G)-sum(G)} negatives")

MODELS = [   # (label, model, extra_body for reasoning models)
    ("yandexgpt-5-lite", yuri("yandexgpt-5-lite"), None),
    ("yandexgpt-5-pro",  yuri("yandexgpt-5-pro"),  None),
    ("yandexgpt-5.1",    yuri("yandexgpt-5.1"),    None),
    ("gpt-oss-120b",     yuri("gpt-oss-120b"),     {"reasoning_effort":"low"}),
    ("qwen3.6-35b-a3b",  yuri("qwen3.6-35b-a3b"),  {"chat_template_kwargs":{"enable_thinking":False}}),
]
BATCHES = [10, 20, 40]

def prf_keep(keep, gold):
    tp=sum(1 for k,y in zip(keep,gold) if k and y==1); fp=sum(1 for k,y in zip(keep,gold) if k and y==0)
    fn=sum(1 for k,y in zip(keep,gold) if (not k) and y==1)
    P=tp/(tp+fp) if tp+fp else 0.0; R=tp/(tp+fn) if tp+fn else 0.0
    return P, R, (2*P*R/(P+R) if P+R else 0.0), tp, fp, fn

rows=[]
for label, uri, extra in MODELS:
    for bs in BATCHES:
        f = LLMGapFilter(backend="api", model=uri, mode="validate_rct", client=CLIENT)
        if extra: f.extra_body = extra
        t0=time.time()
        try:
            keep=[]
            for k in range(0,len(S),bs): keep.extend(f.judge_batch(S[k:k+bs]))
            P,R,F1,tp,fp,fn = prf_keep(keep, G)
            rows.append(dict(model=label,batch=bs,P=round(P,3),R=round(R,3),F1=round(F1,3),
                             calls=f.n_calls,secs=round(time.time()-t0,1)))
        except Exception as e:
            rows.append(dict(model=label,batch=bs,P=None,R=None,F1=None,calls=None,secs=None,err=type(e).__name__))
probe = pd.DataFrame(rows); probe

### Pick
On the probe, batch **20** holds accuracy at ~1/3 the calls, and **yandexgpt-5-lite** / **5-pro**
top F1. We default to **yandexgpt-5-lite @ batch 20** (cheapest strong option); flip `WINNER` to
`yandexgpt-5-pro` for max robustness.

> Caveats: tiny sample (±1 sentence ≈ 1.7% F1 — the top models are within noise), and it's *easy*
> (clean gold vs clean negatives). Real Stage C sees Stage-B **hard** positives, so the numbers in
> Experiment 2 (below) are the honest ones.

## Experiment 2 — the real metric: Stage B detector → Stage C, on SAL

Now the honest test. We train a **Stage B** detector (frozen bge-small + logreg) on SAL-train, run it
on SAL-test to get gap *candidates*, then apply **Stage C** over the positives. Because precision is
prevalence-dependent, we report two operating points:
- **full stream** (~2.6% prevalence) — realistic needle-in-haystack;
- **prevalence-matched to 20.7%** — apples-to-apples vs Lan et al. (2024), PubMedBERT **F1 0.821**.

> `TEST_CAP` keeps a smoke run fast; set it to `None` and **Run All** for the full-stream number.

In [ ]:
WINNER = yuri("yandexgpt-5-lite")   # <- flip to yuri("yandexgpt-5-pro") for robustness
CHUNK  = 20
TEST_CAP = 1500   # None = full 7,968-sentence test stream (slower, full metric)

from bench_rct import load as sal_load
from bench_limgen import prf
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

tr_s, tr_y = sal_load("train")
te_s, te_y = sal_load("test", TEST_CAP)
print(f"train {len(tr_s)} ({int(tr_y.sum())} lim) | test {len(te_s)} ({int(te_y.sum())} lim, {te_y.mean():.1%})")

enc = SentenceTransformer("BAAI/bge-small-en-v1.5")
emb = lambda xs: enc.encode(xs, normalize_embeddings=True, batch_size=128, show_progress_bar=False)
clf = LogisticRegression(C=2, max_iter=2000, class_weight="balanced").fit(emb(tr_s), tr_y)
sb = clf.predict(emb(te_s))      # Stage B positives (recall-first)
print("Stage B positives:", int(sb.sum()), "of", len(sb))

In [ ]:
# Stage C over Stage-B positives
def run_stage_c(sents, sb, model=WINNER, chunk=CHUNK, mode="validate_rct"):
    f = LLMGapFilter(backend="api", model=model, mode=mode, client=CLIENT)
    pos = [i for i,v in enumerate(sb) if v==1]
    keep=[]
    for k in range(0,len(pos),chunk): keep.extend(f.judge_batch([sents[i] for i in pos[k:k+chunk]]))
    final = sb.copy()
    for i,kp in zip(pos,keep):
        if not kp: final[i]=0
    return final, f.n_calls, len(pos), sum(1 for x in keep if not x)

t0=time.time()
sc, ncalls, njudged, ndrop = run_stage_c(te_s, sb)
print(f"Stage C: judged {njudged} positives in {ncalls} calls, dropped {ndrop}  [{time.time()-t0:.0f}s]")

def line(name, pred, y):
    P,R,F = prf(y, pred); return dict(stage=name, P=P, R=R, F1=F, flagged=int(np.sum(pred)))
full = pd.DataFrame([line("Stage B only", sb, te_y), line("+ Stage C", sc, te_y)]).set_index("stage")
print("\\n--- full stream (~2.6% prevalence) ---"); print(full)

In [ ]:
# prevalence-matched to 20.7% (vs Lan PubMedBERT 0.821): keep all positives, subsample negatives
def prevalence_match(y, target=0.207, seed=0):
    pos=np.where(y==1)[0]; neg=np.where(y==0)[0]
    n_neg=int(round(len(pos)*(1-target)/target))
    rng=np.random.RandomState(seed); neg=rng.permutation(neg)[:n_neg]
    idx=np.sort(np.concatenate([pos,neg])); return idx

idx = prevalence_match(te_y)
pm = pd.DataFrame([line("Stage B only", sb[idx], te_y[idx]),
                   line("+ Stage C", sc[idx], te_y[idx])]).set_index("stage")
pm.loc["Lan 2024 PubMedBERT (SOTA)"] = {"P":0.751,"R":0.907,"F1":0.821,"flagged":np.nan}
print(f"--- prevalence-matched to {te_y[idx].mean():.1%} (vs their 20.7%) ---"); pm

# NOTE: negatives here are sampled from the WHOLE stream (easy). Their pool is section-filtered
# (harder negatives). For the exact pool, run scripts/bench/build_rct_pool.py (PMC BioC API).

## Findings

- **Stage C is a precision filter:** it lifts precision by dropping prior-work critiques / contribution
  claims, at a small recall cost. The net F1 gain shows up **only where there are true negatives** (here).
- On the **prevalence-matched** pool the detector + Stage C approaches the Lan 2024 PubMedBERT SOTA
  (0.821) — using a frozen, general-domain head + an LLM filter, no biomedical fine-tune.
- **Cost:** Stage C judged only the Stage-B positives, in a handful of batched calls — the whole point
  of putting it last in the funnel.

> Numbers depend on `TEST_CAP` (smoke vs full) and the chosen `WINNER` model. Run All with
> `TEST_CAP=None` for the headline figure.